In [19]:
# to hide authentication strings
import getpass as gp

# for managing connections
from teradataml import create_context, get_context, remove_context

# for setting configure options
from teradataml import configure

# DataFrames
from teradataml.dataframe.dataframe import DataFrame, in_schema

# for dropping tables or views
from teradataml.dbutils.dbutils import db_drop_table, db_drop_view 

import pandas as pd
import numpy as np
from teradatasqlalchemy.types import *
from sqlalchemy import text 

# for fastload, copy_to_sql, fastexport
from teradataml.dataframe.fastload import fastload
from teradataml.dataframe.data_transfer import fastexport
from teradataml.dataframe.copy_to import copy_to_sql

In [58]:
df = pd.read_excel('MTDT_vx.xlsx', sheet_name='TBL')


In [59]:
df.head(2)

,TBL_ID,TBL_NM,TBL_BUS_NM,TBL_BUS_DESC,AST_ID,DB_ID,DB_OWNR_ID,SBJ_AREA_ID,TBL_SRC_TXT,TMLN_IND,TBL_TYP_TXT,UPDT_FREQ_TXT,USER_UPDT_ID,USER_UPDT_TS,LST_UPDT_BTCH_ID,LST_UPDT_TS
0,1,ACCT_PAY_RMRK_RSN_TYP,ACCOUNT PAYABLE REMARK REASON TYPE,The Accounts Payable Remark Reason Type table ...,1,1,1,15,NaN,N,Fact,Daily,mkhanna8@Ultamcloud.com,2025-03-21 09:28:41.637,NaN,NaT
1,2,ACCT_RECV_TYP,ACCOUNT RECEIVABLE TYPE,Account Receivable Type ACCT RECV TYP is a ref...,1,1,1,15,NaN,N,Fact,Daily,uwhbeutp,2021-12-17 13:48:06.540,6608116.0,2021-12-17 13:48:06.540


In [47]:
td_context = create_context(host = 'tddevtest.td.teradata.com', 
                            username= 'js186127', 
                            password = 'BombadeSumider0$', 
                            logmech='LDAP', 
                            sslmode='ALLOW', 
                            database='ADLSTE_HCLS_MCP_DATA_DICT')

c:\Users\js186127\AppData\Local\Programs\Python\Python312\Lib\site-packages\teradatasqlalchemy\telemetry\queryband.py:382: UserWarning: [Teradata][teradataml](TDML_2002) Overwriting an existing context associated with Teradata Vantage Connection. Most of the operations on any teradataml DataFrames created before this will not work.
  return exposed_func(*args, **kwargs)


In [60]:
from teradataml import create_context, get_context, remove_context, get_connection
conn = get_connection()
print(conn)

In [61]:

conn.execute(text('''delete from ADLSTE_HCLS_MCP_DATA_DICT.TBL;'''))

In [62]:
df.dtypes

TBL_ID                       int64
TBL_NM                      object
TBL_BUS_NM                  object
TBL_BUS_DESC                object
AST_ID                       int64
DB_ID                        int64
DB_OWNR_ID                   int64
SBJ_AREA_ID                  int64
TBL_SRC_TXT                 object
TMLN_IND                    object
TBL_TYP_TXT                 object
UPDT_FREQ_TXT               object
USER_UPDT_ID                object
USER_UPDT_TS        datetime64[ns]
LST_UPDT_BTCH_ID           float64
LST_UPDT_TS         datetime64[ns]
dtype: object

In [63]:
# Fix ALL object columns - convert float NaN to None
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].apply(lambda x: str(x) if pd.notna(x) else None)

In [64]:
df['USER_UPDT_TS'] = df['USER_UPDT_TS'].astype(str)
df['LST_UPDT_TS'] = df['LST_UPDT_TS'].astype(str)
df['LST_UPDT_BTCH_ID'] = df['LST_UPDT_BTCH_ID'].astype(str)

In [65]:
df.dtypes

TBL_ID               int64
TBL_NM              object
TBL_BUS_NM          object
TBL_BUS_DESC        object
AST_ID               int64
DB_ID                int64
DB_OWNR_ID           int64
SBJ_AREA_ID          int64
TBL_SRC_TXT         object
TMLN_IND            object
TBL_TYP_TXT         object
UPDT_FREQ_TXT       object
USER_UPDT_ID        object
USER_UPDT_TS        object
LST_UPDT_BTCH_ID    object
LST_UPDT_TS         object
dtype: object

            types={'TBL_ID' : INTEGER,
                   'TBL_NM' : VARCHAR(255),
                   'TBL_CD' : VARCHAR(255),
                   'TBL_CD2' : VARCHAR(255),
                   'TBL_BUS_NM' : VARCHAR(255)
            }

In [67]:
# for defining SQL data types
from teradatasqlalchemy.types import *

copy_to_sql(df = df, 
            table_name = 'TBL', 
            schema_name = 'ADLSTE_HCLS_MCP_DATA_DICT', 
            temporary = False           # not volatile/temp table

           )